# Overview of the Encoder Architecture

The encoder architecture is transfomer based with linear attention, SwiGLU ad RMSNorm.  This notebook will walk through each layer so that the reader can build an intuition to what each layer is doing to the data.  To this extent, you'll see that we set the layer initializations and numbers to ones where you can hand calculate if you need to follow a layer better. 

In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import math

## Data Prep

We'll start by a simple data prep.  Here we'll use a small batch of 2 samples, each with 8 gene expression counts. 

In [2]:
batch = 2 # Batch
num_genes = 8 # context, aka num of genes


SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
x_input = torch.from_numpy(np.round(np.random.uniform(1, 5, size=(batch, num_genes)), 0)).float() # [batch, num_genes]
total_counts = torch.from_numpy(np.random.randint(1, 10, size=(batch))).float()# [batch]

x_input.shape, x_input, total_counts.shape, total_counts

(torch.Size([2, 8]),
 tensor([[2., 2., 2., 3., 2., 3., 2., 5.],
         [4., 1., 3., 4., 2., 5., 3., 4.]]),
 torch.Size([2]),
 tensor([2., 4.]))

## Data Masking 
(only done for the Context/Student) We create a mask to make the target prediction task harder.  we do evaluate only the masked positions when we determine loss. We'll first generate a random distribution and then everything below our target masking threshold will be masked. For the sake fo the demonstration I'll use a lower masking ratio than our model> 

In [4]:
mask_ratio = 0.4 
rand = torch.rand(batch, num_genes)
rand

tensor([[0.0783, 0.4956, 0.6231, 0.4224, 0.2004, 0.0287, 0.5851, 0.6967],
        [0.1761, 0.2595, 0.7086, 0.5809, 0.0574, 0.7669, 0.8778, 0.2434]])

In [5]:
mask_idx = rand < mask_ratio #use probabilistic masking. it's not perfect but will work well over a large training run
mask_idx

tensor([[ True, False, False, False,  True,  True, False, False],
        [ True,  True, False, False,  True, False, False,  True]])

now we'll apply the mask.  I'll make a copy of our `x_values` so we can see how the masking changes.  

In [6]:
x_values = x_input.clone()
x_values[mask_idx] = 0.0
x_values.shape, x_values

(torch.Size([2, 8]),
 tensor([[0., 2., 2., 3., 0., 0., 2., 5.],
         [0., 0., 3., 4., 0., 5., 3., 0.]]))

## Forward Pass

We start by inserting in a channels dimension.  Right now we just have 1 value per gene, but we'll represent each gene with many dimensions `embed_dim` to let the model learn different combinations of gene importants.  We'll also use multiple `heads` which is basically a grouping of the embedding dimension channels so that combinations of them can learn different complex topics. 

In [7]:
embed_dim = 6
heads = 3

**Insert in the channels dimension**

In [8]:
x = x_values.unsqueeze(-1)
x.shape, x

(torch.Size([2, 8, 1]),
 tensor([[[0.],
          [2.],
          [2.],
          [3.],
          [0.],
          [0.],
          [2.],
          [5.]],
 
         [[0.],
          [0.],
          [3.],
          [4.],
          [0.],
          [5.],
          [3.],
          [0.]]]))

### Fourier FiLM gene encoding

Instead of just relying on gene expression counts, we want to do a Fourier FiLM based projection of the gene expression counts with learnable controls on the projection.  We do this since we know that in biology, expression is typically non-linear where you have different expression plateaus including fully on or off. In this projection expression values are first sclaed by a learned gene embeddings and a scaler, then a random Fourier feature network generates expression dependent FiLM parameters (gamma, beta) to further modulate the result based on expression level. The two paths give the model both a linear signal (expression * embedding) and a nonlinear one (Fourier features -> FiLM), combined into the final gene representation. The goal is to apply the following

$\mathbf{h}_g = \alpha \, x_g \, \mathbf{e}_g \odot (1 + \boldsymbol{\gamma}_g) + \boldsymbol{\beta}_g$

where

$[\boldsymbol{\gamma}_g, \boldsymbol{\beta}_g] = \mathrm{MLP}(\phi(\alpha' x_g))$

- $x_g$ is the expression value for gene $g$
- $\mathbf{e}_g$ is the learned gene embedding
- $\alpha, \alpha'$ are learned scalars
- $\phi$ is a random Fourier feature mapping
- $\odot$ is elementwise multiplication

While we use an multi layer perceptron (MLP), the FiLM portion is specifically the $\mathbf{h}_g = x * (1 + \gamma) + \beta$ pattern. This becomes an affine transformation where gamma and beta are conditioned on some input. 

#### $\alpha$ Gene Expression Count Scaling

We'll start with our gene expression count scaler.  This will be a single value that we'll mutliply against our scaled gene expression count embeddings. We use a linear layer here so that backprop can update this scaler.  We'll first setup the learned scaler and multiply it by expression counts, after which we'll then use that to scale our gene embeddings. We'll initilize this scaler to `1.5` so you'll see that our initial expression values grow.

In [9]:
expr_scaler = nn.Linear(1, 1, bias=False)
nn.init.constant_(expr_scaler.weight, 1.5)
expr_scaler.weight

Parameter containing:
tensor([[1.5000]], requires_grad=True)

In [10]:
scaled_x = expr_scaler(x)
scaled_x.shape, scaled_x

(torch.Size([2, 8, 1]),
 tensor([[[0.0000],
          [3.0000],
          [3.0000],
          [4.5000],
          [0.0000],
          [0.0000],
          [3.0000],
          [7.5000]],
 
         [[0.0000],
          [0.0000],
          [4.5000],
          [6.0000],
          [0.0000],
          [7.5000],
          [4.5000],
          [0.0000]]], grad_fn=<UnsafeViewBackward0>))

#### $\mathbf{e}_g$ Gene Embeddings

Now we'll initialize a representation of the gene embeddings and then scale them based on our scaled expression counts. You'll see that the masking extends across all embedding channels. Also, since we used an initialization where each channel has the same value, you'll see that that multiple creates consistent channel entries per gene

In [11]:
# creates an incremental weight for easier following
vs, d = num_genes, embed_dim
rows = torch.arange(vs).unsqueeze(1)
cols = torch.full((d,), 1.0).unsqueeze(0)
pattern = 0.1*(rows + cols)  

gene_embeddings = nn.Parameter(pattern)
gene_embeddings

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.3000, 0.3000, 0.3000, 0.3000, 0.3000, 0.3000],
        [0.4000, 0.4000, 0.4000, 0.4000, 0.4000, 0.4000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000],
        [0.7000, 0.7000, 0.7000, 0.7000, 0.7000, 0.7000],
        [0.8000, 0.8000, 0.8000, 0.8000, 0.8000, 0.8000]], requires_grad=True)

In [12]:
scaled_x = gene_embeddings.unsqueeze(0) * scaled_x
scaled_x

tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000],
         [0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000],
         [1.8000, 1.8000, 1.8000, 1.8000, 1.8000, 1.8000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [2.1000, 2.1000, 2.1000, 2.1000, 2.1000, 2.1000],
         [6.0000, 6.0000, 6.0000, 6.0000, 6.0000, 6.0000]],

        [[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [1.3500, 1.3500, 1.3500, 1.3500, 1.3500, 1.3500],
         [2.4000, 2.4000, 2.4000, 2.4000, 2.4000, 2.4000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [4.5000, 4.5000, 4.5000, 4.5000, 4.5000, 4.5000],
         [3.1500, 3.1500, 3.1500, 3.1500, 3.1500, 3.1500],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]]],
       grad_fn=<MulBackward0>)

#### $\alpha'$ Fourier Scaler 

Now we move on to the fourier half of the calculation.  We'll start by calculating the scaler. Again we use a linear here so that it represents a learnable parameter that will drift during backprop. To make sure the values diverge from the expression scaler, I'll use different value `0.5`.  When applied, you can see our initial expression values are cut in half. 

In [13]:
fourier_input_scaler = nn.Linear(1, 1, bias=False)
nn.init.constant_(fourier_input_scaler.weight, 0.5)
fourier_input_scaler.weight

Parameter containing:
tensor([[0.5000]], requires_grad=True)

In [14]:
fourier_x = fourier_input_scaler(x)
fourier_x.shape, fourier_x

(torch.Size([2, 8, 1]),
 tensor([[[0.0000],
          [1.0000],
          [1.0000],
          [1.5000],
          [0.0000],
          [0.0000],
          [1.0000],
          [2.5000]],
 
         [[0.0000],
          [0.0000],
          [1.5000],
          [2.0000],
          [0.0000],
          [2.5000],
          [1.5000],
          [0.0000]]], grad_fn=<UnsafeViewBackward0>))

#### $\phi$ Fourier Projection  
Now we'll do the fourier projection. This step projects the scaled expression counts through a fixed random matrix `fp_scaler`. This random matrix is half the size since we take the sin and cos of the result and then concatenate them together to produce a full-dimensional embedding. 

This step maps a the continuous expression level into a rich high-dimensional representation where nearby values have similar features allowing the model to learn representations from when the different plateaus of expession levels.  The `gaussian_scale` scaler is a tunable hyperparameter that can quickly improve and degrade model performance.

Since we're working with sine/cosine we have to think of the periodicity.  We first will take our expression and spread them across a full sin/cosine cycle, which, if you remember your trig, is $2\pi$.  This normalizes the input so that a unit change in the input corresponds to one full cycle of sin/cos. If we did not do this, the random projection matrix alone controls the frequency, increasing the fragility of the projection.

**Numeric computing** one thing to remember is that we're working with fixed precision.  Because of this, when we use irrational numbers like $\pi$, we must store a numeric representation meaning only a certain precision is stored. This means that while we should expect integer representations for multiple of $pi$, we actually sometimes get infinitesimal instead. While it's not the most precise, this random introduced noise will just act as a mini-bias to our model.  Take the bewlo example, for `sin` we should see `[0,1,0,-1,0]` and for `cos` we should see `[1,0,-1,0,1]` but instead we see some values not quite there.  Keep this in mind as we'll see this numeric computing error introduced in our notebook. 

In [15]:
examp_array = torch.tensor([0,0.25,0.5,0.75,1]).float()
torch.sin(examp_array*2*np.pi), torch.cos(examp_array*2*np.pi)

(tensor([ 0.0000e+00,  1.0000e+00, -8.7423e-08, -1.0000e+00,  1.7485e-07]),
 tensor([ 1.0000e+00, -4.3711e-08, -1.0000e+00,  1.1925e-08,  1.0000e+00]))

In [16]:
x_fp = (2 * np.pi * fourier_x)
x_fp

tensor([[[ 0.0000],
         [ 6.2832],
         [ 6.2832],
         [ 9.4248],
         [ 0.0000],
         [ 0.0000],
         [ 6.2832],
         [15.7080]],

        [[ 0.0000],
         [ 0.0000],
         [ 9.4248],
         [12.5664],
         [ 0.0000],
         [15.7080],
         [ 9.4248],
         [ 0.0000]]], grad_fn=<MulBackward0>)

Now that we've scaled fourier based our expression counts by $2\pi$, we'll inject half of the channels. As part of this injection, we use random initiated noise for each channel and a tunable hyperparameter scaler `gaussian_scale` to increment the noise.  For learning, we'll keep our initiation preset.  

That said, you'll see that we do not use gradiants as this is not learnable. Our goal with the random Fourier features is that a fixed random projection is theoretically sufficient to approximate a shift-invariant kernel (we care about differences in expression values, not the actual numeric number). If we made it learnable, the network would likely collapse the frequencies to overfit or degenerate, defeating the purpose of providing a diverse multi-frequency basis. The nonlinear expressiveness of this layer comes downstream with the film generation MLP that processes the Fourier features.  That layer is learnable.

In [17]:
gaussian_scale = 2.0
fp_scaler = nn.Parameter(torch.tensor([[0.5,1.0,1.5]])* gaussian_scale, requires_grad=False)
fp_scaler

Parameter containing:
tensor([[1., 2., 3.]])

In [18]:
x_fp = x_fp @ fp_scaler
x_fp.shape, x_fp

(torch.Size([2, 8, 3]),
 tensor([[[ 0.0000,  0.0000,  0.0000],
          [ 6.2832, 12.5664, 18.8496],
          [ 6.2832, 12.5664, 18.8496],
          [ 9.4248, 18.8496, 28.2743],
          [ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000],
          [ 6.2832, 12.5664, 18.8496],
          [15.7080, 31.4159, 47.1239]],
 
         [[ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000],
          [ 9.4248, 18.8496, 28.2743],
          [12.5664, 25.1327, 37.6991],
          [ 0.0000,  0.0000,  0.0000],
          [15.7080, 31.4159, 47.1239],
          [ 9.4248, 18.8496, 28.2743],
          [ 0.0000,  0.0000,  0.0000]]], grad_fn=<UnsafeViewBackward0>))

**Sin/Cos**

Now we're read for our radial extraction to take the sine and cosine.  As a reminder, because we're dealing with numeric computing, instead of nice clean integers we'll see that we have infinitesimal introduced.  Interestingly, this is more prefelant in sine vs cosine bringing us back up to our embedding dimension size. 

In [19]:
x_fp_sin = torch.sin(x_fp)
x_fp_sin

tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 1.7485e-07,  3.4969e-07,  4.7700e-08],
         [ 1.7485e-07,  3.4969e-07,  4.7700e-08],
         [-2.3850e-08,  4.7700e-08, -7.1549e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 1.7485e-07,  3.4969e-07,  4.7700e-08],
         [-6.7553e-07,  1.3511e-06, -3.9339e-06]],

        [[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [-2.3850e-08,  4.7700e-08, -7.1549e-08],
         [ 3.4969e-07,  6.9938e-07,  9.5399e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [-6.7553e-07,  1.3511e-06, -3.9339e-06],
         [-2.3850e-08,  4.7700e-08, -7.1549e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00]]], grad_fn=<SinBackward0>)

In [20]:
x_fp_cos = torch.cos(x_fp)
x_fp_cos

tensor([[[ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [-1.,  1., -1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [-1.,  1., -1.]],

        [[ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [-1.,  1., -1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [-1.,  1., -1.],
         [-1.,  1., -1.],
         [ 1.,  1.,  1.]]], grad_fn=<CosBackward0>)

In [21]:
fourier_x = torch.cat([x_fp_sin, x_fp_cos], dim=-1)
fourier_x.shape, fourier_x

(torch.Size([2, 8, 6]),
 tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 1.7485e-07,  3.4969e-07,  4.7700e-08,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 1.7485e-07,  3.4969e-07,  4.7700e-08,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [-2.3850e-08,  4.7700e-08, -7.1549e-08, -1.0000e+00,  1.0000e+00,
           -1.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 1.7485e-07,  3.4969e-07,  4.7700e-08,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [-6.7553e-07,  1.3511e-06, -3.9339e-06, -1.0000e+00,  1.0000e+00,
           -1.0000e+00]],
 
         [[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  

#### $\mathrm{MLP}$ Multilayer perceptron  

Now we're ready to add the nonlinear expressiveness for the the film generator by using a MLP. The MLP will provide a learnable linear layer, a nonlinearity, and a final upward projection into a doubling to create our $\gamma$ and $\beta$ for our FiLM calcuation. These learnable layers are what allow the model to decide how much and which parts of the Fourier Projection we want to include with our initial scaled expression. 

**MLP - linear 1** We'll first start with a single linear layer that allows the model to decide how much each channel learned should interact with the other.  Recall that half our channels are sine, and half are cosine, so this allows the model to mix the two radial projections. 

*You'll notice that near 0 values here are washed out*

In [22]:
mlp_l1 = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(mlp_l1.weight, .25)
nn.init.constant_(mlp_l1.bias, 0.0)
mlp_l1.weight

Parameter containing:
tensor([[0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500]], requires_grad=True)

In [23]:
fourier_x = mlp_l1(fourier_x)
fourier_x.shape, fourier_x

(torch.Size([2, 8, 6]),
 tensor([[[ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500]],
 
         [[ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0

**MLP - GELU nonlinearity** Now we're ready for our non-linearity. The [GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html) function is approximately linear above 1 and pulls most values below -2 to 0.  Between -2 and 0, most values are pulled closer to 0 and there's a slight non-linearity between 0 and 1.  Since we used sine/cosine values, most of our values at this point should be between 0 and 1 so we'll benefit from the non-linear portion with a cap on highly negative values. 

In [24]:
mlp_gelu = nn.GELU()

fourier_x = mlp_gelu(fourier_x)
fourier_x.shape, fourier_x

(torch.Size([2, 8, 6]),
 tensor([[[ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003]],
 
         [[ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0

**MLP - linear 2** Now we'll scale up to 2x our embedding size since we need to provide values for our two variables in our FiLM projection. We use a learnable linear scale up layer so that the model can learn how to split values across the two variables.  For our initiation we'll increment the second half to be twice the first half to show the differences. 

In [25]:
mlp_l2 = nn.Linear(embed_dim, embed_dim*2)
nn.init.constant_(mlp_l2.weight[:embed_dim, :], 0.5)
nn.init.constant_(mlp_l2.weight[embed_dim:, :], 1.0)
nn.init.constant_(mlp_l2.bias, 0.0)
mlp_l2.weight.shape, mlp_l2.weight

(torch.Size([12, 6]),
 Parameter containing:
 tensor([[0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]], requires_grad=True))

In [26]:
fourier_x = mlp_l2(fourier_x)
fourier_x.shape, fourier_x

(torch.Size([2, 8, 12]),
 tensor([[[ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.6019,
           -0.6019, -0.6019, -0.6019, -0.6019, -0.6019],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0

#### $\gamma_g, \beta_g$ FiLM components 
Now that we have the MLP output, we're ready to build our FiLM variables.  Recall that the for FiLM we calculate $\mathbf{h}_g = x * (1 + \gamma) + \beta$ where $x$ will be the weighted scaler of the expression counts. To get $\gamma$ and $\beta$ we simly split the output of the MLP.  Recall that we built it so that half of the MLP output was doubled so when we split, we should see that $\beta$ is double $\gamma$.

In [27]:
gamma, beta = torch.chunk(fourier_x, 2, dim=-1)
gamma.shape, gamma, beta.shape, beta

(torch.Size([2, 8, 6]),
 tensor([[[ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010]],
 
         [[ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0

#### $\mathbf{h}_g$ FiLM based gene expression respresentation 

Now we're ready for the final FiLM calculation.  FiLM can be thought of as summing two weighted parts, in our case a scaled version of the expression counts and a radial projection of the expression counts. What's interesting is that the FiLM formula pushes the radial projection both as a scaler to the initial counts and a bias similar to $x = mx + b$.  Ultimately we've given the model the ability to learn how to upscale the original counts, shift the counts based on the radial projection, and add/subtract the counts based on the raidal projection.  All of this allows the model to learn a more complex landscape to scale gene embeddings by the expression beyond the typical linear scaling where 2 expression counts mean double 1 expression. 

In [28]:
x = scaled_x * (1.0 + gamma) + beta

x.shape, x

(torch.Size([2, 8, 6]),
 tensor([[[ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 5.1242,  5.1242,  5.1242,  5.1242,  5.1242,  5.1242],
          [ 5.9463,  5.9463,  5.9463,  5.9463,  5.9463,  5.9463],
          [ 0.6563,  0.6563,  0.6563,  0.6563,  0.6563,  0.6563],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 9.2344,  9.2344,  9.2344,  9.2344,  9.2344,  9.2344],
          [ 3.5922,  3.5922,  3.5922,  3.5922,  3.5922,  3.5922]],
 
         [[ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 0.3417,  0.3417,  0.3417,  0.3417,  0.3417,  0.3417],
          [10.0564, 10.0564, 10.0564, 10.0564, 10.0564, 10.0564],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 2.5437,  2.5437,  2.5437,  2.5437,  2.5437,  2.5437],
          [ 1.6000,  1.6000,  1.6000,  1.6000,  1

#### Remask with learned mask

Now we need to reintroduce the masking back but, instead of a 0 value, we want to actually let the model learn a mask token.  We learn a mask token because the model needs to distinguish "this gene is masked and I need to predict it" from "this gene has zero expression." If the mask token were fixed (e.g. all zeros), it would be indistinguishable from a zero-expression gene's representation after the Fourier FiLM encoding. A learned token lets the model settle on a representation that optimally signals "predict me" to the downstream transformer blocks. Ultimately we're reapplying the masking at this point mainly so that we do our expression encoding cleanly first, and then maks after. Since our mask token is learnable, instead of a common `-1` hardcode, the initation in the model will use random learnable values.  What we'll do in our example is use `-11` so it really sticks out.  These values will change during backprop as the model learns what a good token is to prompt it that it needs replacement. 

*As a reminder, masking only happens on the context encoder, and not the target encoder during our models forward pass*

In [29]:
mask_token = nn.Parameter(torch.randn(embed_dim) * 0.02)
nn.init.constant_(mask_token, -11)
mask_token

Parameter containing:
tensor([-11., -11., -11., -11., -11., -11.], requires_grad=True)

In [30]:
x = torch.where(mask_idx.unsqueeze(-1), mask_token, x)
x

tensor([[[-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  5.1242,   5.1242,   5.1242,   5.1242,   5.1242,   5.1242],
         [  5.9463,   5.9463,   5.9463,   5.9463,   5.9463,   5.9463],
         [  0.6563,   0.6563,   0.6563,   0.6563,   0.6563,   0.6563],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  9.2344,   9.2344,   9.2344,   9.2344,   9.2344,   9.2344],
         [  3.5922,   3.5922,   3.5922,   3.5922,   3.5922,   3.5922]],

        [[-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  0.3417,   0.3417,   0.3417,   0.3417,   0.3417,   0.3417],
         [ 10.0564,  10.0564,  10.0564,  10.0564,  10.0564,  10.0564],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  2.5437,   2.5437,   2.5437,   2.5437,   2.5437,   2.5437],
    

### Total count injection

Now we'll want to add the total count in. In our data prep, we normalize all cell expression counts to the same total count.  This is great since perturbSeq is relativistic in it's data, but this normalization has one drawback: cells with abnormally high or low expression totals compared to their peers lose the signal. In particular the abnormally low is our biggest concern as a perturbation that makes a cell barely viable may have very low across the board expression that gets amplified when you bring it up to our normalized count.   To avoid this we add in a learnable weight to the total expression count so that the model can learn expression levels.

The total count is multiplied across the embedding dimensions with a learnable weight and then summed to our gene expression projections. This allows the total count to act almost like a bias term. 

We start by taking our total count, injecting the embedding dimensions, and then shaping it to match our embedding counts. 

In [31]:
x_total_ct = total_counts.unsqueeze(-1)
x_total_ct.shape, x_total_ct

(torch.Size([2, 1]),
 tensor([[2.],
         [4.]]))

In [32]:
total_count_proj = nn.Linear(1, embed_dim)
nn.init.constant_(total_count_proj.weight, 0.1)
nn.init.zeros_(total_count_proj.bias)
total_count_proj.weight

Parameter containing:
tensor([[0.1000],
        [0.1000],
        [0.1000],
        [0.1000],
        [0.1000],
        [0.1000]], requires_grad=True)

In [33]:
x_total_ct = total_count_proj(x_total_ct)
x_total_ct = x_total_ct.unsqueeze(1)
x_total_ct.shape, x_total_ct

(torch.Size([2, 1, 6]),
 tensor([[[0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000]],
 
         [[0.4000, 0.4000, 0.4000, 0.4000, 0.4000, 0.4000]]],
        grad_fn=<UnsqueezeBackward0>))

### Unified representation of the cell state

Now that we have an ebedding representation of both the gene expression and total count, we're ready to sum them for a single representation of the cell state. Now it will be ready for a cell state block

In [34]:
x = x + x_total_ct
x.shape, x

(torch.Size([2, 8, 6]),
 tensor([[[-10.8000, -10.8000, -10.8000, -10.8000, -10.8000, -10.8000],
          [  5.3242,   5.3242,   5.3242,   5.3242,   5.3242,   5.3242],
          [  6.1463,   6.1463,   6.1463,   6.1463,   6.1463,   6.1463],
          [  0.8563,   0.8563,   0.8563,   0.8563,   0.8563,   0.8563],
          [-10.8000, -10.8000, -10.8000, -10.8000, -10.8000, -10.8000],
          [-10.8000, -10.8000, -10.8000, -10.8000, -10.8000, -10.8000],
          [  9.4344,   9.4344,   9.4344,   9.4344,   9.4344,   9.4344],
          [  3.7922,   3.7922,   3.7922,   3.7922,   3.7922,   3.7922]],
 
         [[-10.6000, -10.6000, -10.6000, -10.6000, -10.6000, -10.6000],
          [-10.6000, -10.6000, -10.6000, -10.6000, -10.6000, -10.6000],
          [  0.7417,   0.7417,   0.7417,   0.7417,   0.7417,   0.7417],
          [ 10.4564,  10.4564,  10.4564,  10.4564,  10.4564,  10.4564],
          [-10.6000, -10.6000, -10.6000, -10.6000, -10.6000, -10.6000],
          [  2.9437,   2.9437,   2.94

### Tranformer Block 
Block repeated for however many layers needed. 


```
class CellStateBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = RMSNorm(config.embed_dim)
        self.attn = BioLinearAttention(config)
        self.ln_2 = RMSNorm(config.embed_dim)
        self.mlp = SwiGLU(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x
        ```

#### RMSNorm 1

For our modern transfomer, we use root mean square normalization, or RMSNorm. RMSNorm calculates the following: 
$$
y = \frac{x}{\sqrt{\frac{1}{n} \sum_{i=1}^{n} x_i^2 + \epsilon}} \cdot \gamma
$$

The main reason we use RMSNorm is that it executes faster and uses less memory than standard layer normalization. This efficiency is achieved by entirely removing the mean-centering calculation, which reduces the total number of arithmetic operations and hardware synchronization steps. Recall that normalization is primarily used for large scale training stability (preventing gradient explosion/vanishing). For deep architectures like ours, the mean of the pre-activation inputs naturally stays close to zero during training so removing the mean-centering operation preserves the critical variance-bounding effect. Also the model learns to absorb any minor activation shifts into the subsequent linear weights or the learned affine parameters. Because of this, we're able to use a more efficient normalization.

Since we reuse RMSNorm 3 times, we'll create a class for it.  in the class you can see that we split out the calculation, first limiting the precision of X, then computing $\frac{x}{\sqrt{\frac{1}{n} \sum_{i=1}^{n} x_i^2 + \epsilon}}$, and finally adding the per channel weights $\cdot \gamma$.

Since our entries have identical values across all channels for a gene, the gene becomes uniform +/-1 depending on the sign (the value is the average so it becomes 1)

In [35]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        x_fp32 = x.float()
        norm = x_fp32 * torch.rsqrt(x_fp32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight).type_as(x)

In [36]:
rms1 = RMSNorm(embed_dim)

rms1.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [37]:
x_norm = rms1(x)
x_norm

tensor([[[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000]],

        [[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [-1.0000, -1.0

#### Linear Attention


$$Y = \left( \sigma(X W_{gate}) \odot \frac{\phi(Q) (\phi(K)^T V)}{\phi(Q) \sum_{j=1}^{T_{kv}} \phi(K_j) + \epsilon} \right) W_c$$

Here $\phi(x) = \text{elu}(x) + 1$ serves as the non-negative feature map. $Q$, $K$, and $V$ represent the standard query, key, and value linear projections. The attention output is element-wise multiplied by a sigmoid gate applied to the original input $X$ before the final linear projection $W_c$. The $\epsilon$ term represents the $1 \times 10^{-6}$ constant used for numerical stability in the denominator.

In [ ]:
B, T_q, C = x_norm.size()
head_dim = embed_dim // heads
B, T_q, C, head_dim

In [ ]:
kv_input = x_norm
kv_input

In [ ]:
T_kv = kv_input.size(1)
T_kv

In [ ]:
q_proj = nn.Linear(embed_dim, embed_dim)
q_proj.weight

In [ ]:
q = q_proj(x_norm).view(B, T_q, heads, head_dim).transpose(1, 2)
q.shape, q

In [ ]:
k_proj = nn.Linear(embed_dim, embed_dim)
k_proj.weight

In [ ]:
k = k_proj(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
k.shape, k

In [ ]:
v_proj = nn.Linear(embed_dim, embed_dim)
v_proj.weight

In [ ]:
v = v_proj(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
v.shape, v

In [ ]:
q = F.elu(q) + 1.0
q

In [ ]:
k = F.elu(k) + 1.0
k

In [ ]:
kv_matmul = k.transpose(-2, -1) @ v
kv_matmul.shape, kv_matmul

In [ ]:
k_sum = k.sum(dim=-2).unsqueeze(-1)
k_sum.shape, k_sum

In [ ]:
z = 1.0 / (q @ k_sum + 1e-6)
z.shape, z

In [ ]:
y = (q @ kv_matmul) * z
y.shape, y

In [ ]:
y = y.transpose(1, 2).contiguous().view(B, T_q, C)
y.shape, y

In [ ]:
gate = nn.Linear(embed_dim, embed_dim)
gate.weight

In [ ]:
y = torch.sigmoid(gate(x_norm)) * y
y.shape, y

In [ ]:
c_proj = nn.Linear(embed_dim, embed_dim)
c_proj.weight

In [ ]:
x_attn = c_proj(y)
x_attn.shape, x_attn

#### Residual Connection

In [ ]:
x = x + x_attn
x.shape, x

#### RMSNorm 2

In [ ]:
weight_2 = nn.Parameter(torch.ones(embed_dim))
eps_2 = 1e-6

weight_2, eps_2

In [ ]:
x_fp32 = x.float()
norm = x_fp32 * torch.rsqrt(x_fp32.pow(2).mean(-1, keepdim=True) + eps_2)
norm

In [ ]:
x_norm2 = (norm * weight_2).type_as(x)
x_norm2

#### SwiGLU MLP

In [ ]:
def _make_divisible(v, divisor=4):
    return max(divisor, int(v + divisor / 2) // divisor * divisor)

In [ ]:
mlp_ratio = 2.0
hidden_dim = _make_divisible(int(embed_dim * mlp_ratio * 2 / 3))
hidden_dim

In [ ]:
w1 = nn.Linear(embed_dim, hidden_dim, bias=False)
w1.weight

In [ ]:
xw1 = w1(x_norm2)
xw1.shape, xw1

In [ ]:
w2 = nn.Linear(embed_dim, hidden_dim, bias=False)
w2.weight

In [ ]:
xw2 = w2(x_norm2)
xw2.shape, xw2

In [ ]:
xw1 = F.silu(xw1)
xw1.shape, xw1

In [ ]:
xw = xw1 * xw2
xw.shape, xw

In [ ]:
w3 = nn.Linear(hidden_dim, embed_dim, bias=False)
w3.weight

In [ ]:
xmlp = w3(xw)
xmlp.shape, xmlp

#### Residual Connection 2

In [ ]:
x = x + xmlp
x.shape, x

### Final Layer Normalization

In [ ]:
weight_f = nn.Parameter(torch.ones(embed_dim))
eps_f = 1e-6

weight_f, eps_f

In [ ]:
x_fp32 = x.float()
norm = x_fp32 * torch.rsqrt(x_fp32.pow(2).mean(-1, keepdim=True) + eps_f)
norm

In [ ]:
x = (norm * weight_f).type_as(x)
x